In [3]:
def print_extended_truth_table():
    # Define symbols (Mapping original A -> P, B -> Q, C -> R)
    symbols = ['P', 'Q', 'R']

    # Print clean headers matching the expanded structure
    print(f" {'P':5} | {'Q':5} | {'R':5} || {'P v R':5} | {'Q v ~R':6} || {'KB':5} || {'alpha':5} | {'R':5} | {'R->P':5} | {'Q->R':5}")
    print("-" * 85)

    # Trackers for logical entailment checks
    entails_alpha = True
    entails_r = True
    entails_r_to_p = True
    entails_q_to_r = True

    n = len(symbols)
    for i in range(2**n):
        # Generate assignments (False first to maintain structure)
        model = {
            'P': bool((i >> 2) & 1),
            'Q': bool((i >> 1) & 1),
            'R': bool((i >> 0) & 1)
        }

        # Core KB sub-expressions
        p_v_r = model['P'] or model['R']
        q_v_not_r = model['Q'] or (not model['R'])
        kb = p_v_r and q_v_not_r

        # Evaluating the target sentences
        alpha = model['P'] or model['Q']
        r = model['R']
        r_to_p = (not model['R']) or model['P']      # Conditional: ~R v P
        q_to_r = (not model['Q']) or model['R']      # Conditional: ~Q v R

        # Entailment verifications (Fails if KB is true but target statement is false)
        if kb and not alpha: entails_alpha = False
        if kb and not r: entails_r = False
        if kb and not r_to_p: entails_r_to_p = False
        if kb and not q_to_r: entails_q_to_r = False

        # Format print rows to lowercase strings
        p_str = str(model['P']).lower()
        q_str = str(model['Q']).lower()
        r_str = str(model['R']).lower()
        pvr_str = str(p_v_r).lower()
        qvnotr_str = str(q_v_not_r).lower()
        kb_str = str(kb).lower()
        alpha_str = str(alpha).lower()
        r_eval_str = str(r).lower()
        r_to_p_str = str(r_to_p).lower()
        q_to_r_str = str(q_to_r).lower()

        print(f" {p_str:5} | {q_str:5} | {r_str:5} || {pvr_str:5} | {qvnotr_str:6} || {kb_str:5} || {alpha_str:5} | {r_eval_str:5} | {r_to_p_str:5} | {q_to_r_str:5}")

    print("-" * 85)
    print(f"i)   Does KB entail Alpha?  -> {entails_alpha}")
    print(f"ii)  Does KB entail R?      -> {entails_r}")
    print(f"iii) Does KB entail R -> P? -> {entails_r_to_p}")
    print(f"iv)  Does KB entail Q -> R? -> {entails_q_to_r}")

if __name__ == "__main__":
    print_extended_truth_table()


 P     | Q     | R     || P v R | Q v ~R || KB    || alpha | R     | R->P  | Q->R 
-------------------------------------------------------------------------------------
 false | false | false || false | true   || false || false | false | true  | true 
 false | false | true  || true  | false  || false || false | true  | false | true 
 false | true  | false || false | true   || false || true  | false | true  | false
 false | true  | true  || true  | true   || true  || true  | true  | false | true 
 true  | false | false || true  | true   || true  || true  | false | true  | true 
 true  | false | true  || true  | false  || false || true  | true  | true  | true 
 true  | true  | false || true  | true   || true  || true  | false | true  | false
 true  | true  | true  || true  | true   || true  || true  | true  | true  | true 
-------------------------------------------------------------------------------------
i)   Does KB entail Alpha?  -> True
ii)  Does KB entail R?      -> False
iii) Doe

In [6]:
import math
import random


def evaluate_cost(state):
    """
    Computes the cost (energy) of the state.
    Cost = the number of pairs of queens attacking each other.
    Target optimal cost is 0.
    """
    conflicts = 0
    n = len(state)

    for i in range(n):
        for j in range(i + 1, n):
            # Check if they share the same row
            if state[i] == state[j]:
                conflicts += 1
            # Check if they share a diagonal
            elif abs(state[i] - state[j]) == abs(i - j):
                conflicts += 1

    return conflicts


def get_random_neighbour(state):
    """
    Generates a neighbor state by picking one random column
    and moving its queen to a different random row.
    """
    neighbour = list(state)
    col = random.randint(0, 7)
    current_row = neighbour[col]

    # Pick a new row that is different from the current row
    new_row = random.randint(0, 7)
    while new_row == current_row:
        new_row = random.randint(0, 7)

    neighbour[col] = new_row
    return neighbour


def simulated_annealing_8_queens(initial_temp=10.0, cooling_rate=0.995, max_iterations=5000):
    """
    Solves the 8-Queens problem strictly following the logic of Algorithm 2.
    """
    # 1: current ← initial state (A random row assignment for each of the 8 columns)
    current_state = [random.randint(0, 7) for _ in range(8)]

    # 2: T ← a large positive value
    T = initial_temp

    iteration = 0
    # 3: while T > 0 do (and cap iterations to prevent infinite loops if no solution is found)
    while T > 0.00001 and iteration < max_iterations:

        # Check if we already found a perfect solution (0 conflicts) to exit early
        if evaluate_cost(current_state) == 0:
            break

        # 4: next ← a random neighbour of current
        next_state = get_random_neighbour(current_state)

        # 5: ΔE ← current.cost - next.cost
        current_cost = evaluate_cost(current_state)
        next_cost = evaluate_cost(next_state)
        delta_E = current_cost - next_cost

        # 6: if ΔE > 0 then (next state has fewer conflicts, so it's a better move)
        if delta_E > 0:
            # 7: current ← next
            current_state = next_state
        # 8: else (next state has more or equal conflicts)
        else:
            # 9: current ← next with probability p = e^(ΔE / T)
            # Since delta_E is negative or zero here, math.exp(delta_E / T) works correctly
            p = math.exp(delta_E / T)
            if random.random() < p:
                current_state = next_state

        # 11: decrease T
        T = T * cooling_rate
        iteration += 1

    # 13: return current
    return current_state


def print_board(state):
    """Visualizes the 8x8 chessboard arrangement."""
    for row in range(8):
        line = ""
        for col in range(8):
            if state[col] == row:
                line += " Q "
            else:
                line += " . "
        print(line)


# =====================================================================
# RUN THE SOLVER
# =====================================================================
if __name__ == "__main__":
    print("--- Searching for an 8-Queens Solution via Simulated Annealing ---")

    # Run the algorithm
    solution = simulated_annealing_8_queens()
    final_cost = evaluate_cost(solution)

    print(f"\nFinal State Representation (Rows per Column): {solution}")
    print(f"Remaining Attacking Pairs (Cost): {final_cost}")

    print("\nVisual Board Layout:")
    print_board(solution)

    if final_cost == 0:
        print("\n✨ Success! A valid 8-Queens configuration was found.")
    else:
        print("\n❌ Timed out before finding a flawless solution. Try running again!")


--- Searching for an 8-Queens Solution via Simulated Annealing ---

Final State Representation (Rows per Column): [4, 6, 0, 2, 7, 5, 3, 1]
Remaining Attacking Pairs (Cost): 0

Visual Board Layout:
 .  .  Q  .  .  .  .  . 
 .  .  .  .  .  .  .  Q 
 .  .  .  Q  .  .  .  . 
 .  .  .  .  .  .  Q  . 
 Q  .  .  .  .  .  .  . 
 .  .  .  .  .  Q  .  . 
 .  Q  .  .  .  .  .  . 
 .  .  .  .  Q  .  .  . 

✨ Success! A valid 8-Queens configuration was found.
